# 🚀 Claude Remote Executor

This notebook creates a **remote execution server** that Claude can connect to from its Python runtime.

**Use cases:**
- Download large datasets (30GB+) that exceed Claude's disk limits
- Run long-running computations without timeout
- Process data that exceeds Claude's memory limits
- Access Google Drive for persistent storage

## Setup Instructions
1. Get a free ngrok auth token from https://ngrok.com (sign up → Your Authtoken)
2. Run all cells below
3. Copy the public URL and share it with Claude
4. Claude can now execute code remotely on this Colab instance!

In [ ]:
# Cell 1: Install dependencies
!pip install fastapi uvicorn pyngrok -q
print("✓ Dependencies installed")

In [ ]:
# Cell 2: Set your ngrok auth token
# Get your free token from: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- Replace this!

# Or use Colab secrets (recommended):
# from google.colab import userdata
# NGROK_AUTH_TOKEN = userdata.get('NGROK_TOKEN')

In [ ]:
# Cell 3: Mount Google Drive (optional but recommended for persistent storage)
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

In [ ]:
# Cell 4: Create the Remote Executor Server

import os
import sys
import json
import traceback
import subprocess
from io import StringIO
from typing import Optional, Dict, Any
from contextlib import redirect_stdout, redirect_stderr

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(
    title="Claude Remote Executor",
    description="Execute Python code remotely from Claude's runtime",
    version="1.0.0"
)

# Allow CORS for Claude's runtime
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Persistent namespace for executed code
execution_namespace = {"__builtins__": __builtins__}

class CodeRequest(BaseModel):
    code: str
    timeout: Optional[int] = 300  # 5 minutes default

class BashRequest(BaseModel):
    command: str
    timeout: Optional[int] = 300

class DownloadRequest(BaseModel):
    url: str
    destination: str = "/content/downloads"
    filename: Optional[str] = None

@app.get("/")
def root():
    return {
        "status": "online",
        "service": "Claude Remote Executor",
        "endpoints": ["/execute", "/bash", "/download", "/ls", "/disk", "/health"]
    }

@app.get("/health")
def health():
    import psutil
    return {
        "status": "healthy",
        "cpu_percent": psutil.cpu_percent(),
        "memory_percent": psutil.virtual_memory().percent,
        "disk_free_gb": round(psutil.disk_usage('/').free / (1024**3), 2)
    }

@app.post("/execute")
def execute_code(request: CodeRequest):
    """Execute Python code and return output + result"""
    stdout_capture = StringIO()
    stderr_capture = StringIO()
    result = None
    error = None
    
    try:
        with redirect_stdout(stdout_capture), redirect_stderr(stderr_capture):
            # Try exec first, then eval for expressions
            try:
                exec(request.code, execution_namespace)
                # Check if last line is an expression
                lines = request.code.strip().split('\n')
                if lines:
                    last_line = lines[-1].strip()
                    if last_line and not any(last_line.startswith(kw) for kw in 
                        ['import', 'from', 'def', 'class', 'if', 'for', 'while', 'with', 'try', '#', 'print']):
                        try:
                            result = eval(last_line, execution_namespace)
                        except:
                            pass
            except SyntaxError:
                result = eval(request.code, execution_namespace)
    except Exception as e:
        error = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
    
    return {
        "success": error is None,
        "stdout": stdout_capture.getvalue(),
        "stderr": stderr_capture.getvalue(),
        "result": repr(result) if result is not None else None,
        "error": error
    }

@app.post("/bash")
def execute_bash(request: BashRequest):
    """Execute bash command and return output"""
    try:
        result = subprocess.run(
            request.command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=request.timeout
        )
        return {
            "success": result.returncode == 0,
            "returncode": result.returncode,
            "stdout": result.stdout,
            "stderr": result.stderr
        }
    except subprocess.TimeoutExpired:
        return {"success": False, "error": f"Command timed out after {request.timeout}s"}
    except Exception as e:
        return {"success": False, "error": str(e)}

@app.post("/download")
def download_file(request: DownloadRequest):
    """Download a file from URL to Colab storage"""
    os.makedirs(request.destination, exist_ok=True)
    
    filename = request.filename or request.url.split('/')[-1].split('?')[0]
    filepath = os.path.join(request.destination, filename)
    
    try:
        result = subprocess.run(
            f'wget -q --show-progress -O "{filepath}" "{request.url}"',
            shell=True,
            capture_output=True,
            text=True,
            timeout=3600  # 1 hour for large files
        )
        
        if os.path.exists(filepath):
            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            return {
                "success": True,
                "filepath": filepath,
                "size_mb": round(size_mb, 2)
            }
        else:
            return {"success": False, "error": "Download failed", "stderr": result.stderr}
    except Exception as e:
        return {"success": False, "error": str(e)}

@app.get("/ls")
def list_directory(path: str = "/content"):
    """List directory contents"""
    try:
        items = []
        for item in os.listdir(path):
            item_path = os.path.join(path, item)
            is_dir = os.path.isdir(item_path)
            size = os.path.getsize(item_path) if not is_dir else None
            items.append({"name": item, "is_dir": is_dir, "size": size})
        return {"success": True, "path": path, "items": items}
    except Exception as e:
        return {"success": False, "error": str(e)}

@app.get("/disk")
def disk_usage():
    """Get disk usage information"""
    import shutil
    total, used, free = shutil.disk_usage("/")
    return {
        "total_gb": round(total / (1024**3), 2),
        "used_gb": round(used / (1024**3), 2),
        "free_gb": round(free / (1024**3), 2),
        "percent_used": round(used / total * 100, 1)
    }

print("✓ FastAPI server created")

In [ ]:
# Cell 5: Start the server with ngrok tunnel (FIXED for Colab's event loop)

import asyncio
import threading
import uvicorn
from pyngrok import ngrok, conf

# Configure ngrok
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill any existing tunnels
ngrok.kill()

PORT = 8000

# Start ngrok tunnel FIRST
public_url = ngrok.connect(PORT, "http").public_url

print("="*60)
print("🚀 CLAUDE REMOTE EXECUTOR IS RUNNING!")
print("="*60)
print(f"\n📡 PUBLIC URL: {public_url}")
print(f"\n📋 Share this URL with Claude to enable remote execution")
print(f"\n📖 API Docs: {public_url}/docs")
print("="*60)
print("\n⚠️  Keep this cell running! The server stops when you stop the cell.")
print("\n")

# Run uvicorn in a separate thread to avoid event loop conflict
def run_server():
    # Create a NEW event loop for this thread
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

# Start server in background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("✓ Server started in background thread")
print(f"\n🔗 Test it: {public_url}/health")

# Keep the cell alive
import time
try:
    while True:
        time.sleep(60)
        print(f"[{time.strftime('%H:%M:%S')}] Server running... URL: {public_url}")
except KeyboardInterrupt:
    print("\n🛑 Server stopped by user")
    ngrok.kill()